In [143]:
import os
import json
import glob
import re
import numpy as np
import xarray as xr
import xskillscore as xs
import matplotlib.pyplot as plt
from typing import Dict
from joblib import Parallel, delayed
from cftime import num2date, DatetimeNoLeap
from datetime import timedelta

In [154]:
class ErrorMetricCalculator:
    def __init__(self, regnam, tstart, tend, frequency,
                 model_list, ref_dict, exp_dict, path_in, out_path,
                 var_list=None, force=False):
        self.regnam = regnam
        self.tstart = tstart
        self.tend = tend
        self.frequency = frequency
        self.model_list = model_list
        self.ref_dict = ref_dict
        self.exp_dict = exp_dict
        self.path_in = path_in
        self.out_path = os.path.join(out_path, frequency)
        self.force = force

        self.var_dict = self.extract_var_list()
        self.var_list = var_list if var_list is not None else list(self.var_dict.keys())
        self.ref_cache = {}
        self.years = list(range(int(self.tstart[:4]), int(self.tend[:4]) + 1))

        self.seasons = {
            "DJF": ([12, 1, 2], "01-03"),
            "MAM": ([3, 4, 5], "04-06"),
            "JJA": ([6, 7, 8], "07-09"),
            "SON": ([9, 10, 11], "10-12"),
        }

        if not os.path.exists(self.out_path):
            os.makedirs(self.out_path)

    def compute(self, metric):
        for year in self.years:
            for var in self.var_list:
                if var not in self.var_dict:
                    raise ValueError(f"Variable '{var}' is not defined in var_dict.")
                vinfo = self.var_dict[var]
                self._compute_seasonal_and_annual_metric(metric, var, vinfo, year)
    # --- Diagnostics ---
    def _summarize_field(self,name, da):
        # Time coverage
        tmin = str(da['time'].min().values) if "time" in da.dims else "N/A"
        tmax = str(da['time'].max().values) if "time" in da.dims else "N/A"
    
        # Level info
        levdim = next((d for d in ("plev", "lev", "level") if d in da.dims), None)
        if levdim:
            levs = da[levdim].values
            lev_units = getattr(da[levdim], "units", "")
            lev_range = f"{levs.min()} – {levs.max()} {lev_units}"
        else:
            lev_range = "no vertical dimension"
    
        return f"{name}: time {tmin} → {tmax}, level {lev_range}"

    def _compute_seasonal_and_annual_metric(self, metric, var, vinfo, year):
        varin = vinfo['alias']
        vfac = vinfo['fscl']
        season_list = ["DJF", "MAM", "JJA", "SON", "ANN"]

        for exp in self.model_list:
            ref = self.exp_dict[exp]['ref']
            out_file = os.path.join(self.out_path, f"{var}_{self.regnam}_{metric}_{exp}_{year}.nc")

            if os.path.exists(out_file):
                if not self.force:
                    print(f"Skipping {metric} for {var} in {exp} ({year}) — file exists.")
                    continue
                else:
                    os.remove(out_file)

            results = {}
            for season in season_list:
                time_sub = self._get_time_range_for_year(year) if season == "ANN" else self._get_time_range_for_season(year, season)
                try:
                    obs = self._read_reference_data(
                        ref=ref,
                        period=f"{year}",
                        time_sub=time_sub,
                        regnam=self.regnam,
                        var=var,
                        vfac=vfac,
                        data_dir=self.ref_dict[ref]['run'],
                        template="monthly/ERA5_analysis_monthly_{}.nc"
                    )[varin].astype("float64")
                    
                    ds = self._read_model_data(
                        exp=exp,
                        period=f"{year}",
                        time_sub=time_sub,
                        regnam=self.regnam,
                        var=var,
                        vfac=vfac,
                        data_dir=self.path_in.replace("%(CASENAME)", self.exp_dict[exp]['run']),
                        template="{}_*.nc"
                    )
                    ds = ds.assign_coords(time=obs['time'])
                    fcst = ds[varin].astype("float64")
                except FileNotFoundError:
                    print(f"Data missing for {var}, {exp}, {year}, {season} — skipping.")
                    results[season] = np.nan
                    continue

                print("Obs range:", float(obs.min().compute()), "to", float(obs.max().compute()))
                print("Fcst range:", float(fcst.min().compute()), "to", float(fcst.max().compute()))
                
                # Rechunk once
                obs  = obs.chunk({'lat': -1, 'lon': -1, 'time': 1})
                fcst = fcst.chunk({'lat': -1, 'lon': -1, 'time': 1})
                
                if metric == "MEAN":
                    val = fcst.weighted(self.generate_coslat_weight(fcst)).mean(("lat", "lon", "time")).values
                elif metric == "BIAS":
                    val = (fcst - obs).weighted(self.generate_coslat_weight(fcst)).mean(("lat", "lon", "time")).values
                elif metric == "RMSE":
                    weights = self.generate_coslat_weight(obs)[0, :, :]
                    val = xs.rmse(obs, fcst, dim=["lat", "lon"], weights=weights, skipna=True).mean("time").values
                elif metric == "ACC":
                    weights = self.generate_coslat_weight(obs)[0, :, :]
                    acc_map = xs.pearson_r(obs - obs.mean("time"), fcst - fcst.mean("time"),
                                           dim="time", skipna=True)
                    val = (acc_map * weights).mean(["lat", "lon"]).values
                elif metric == "STD":
                    val = fcst.std("time").weighted(self.generate_coslat_weight(fcst)).mean(("lat", "lon")).values
                elif metric == "TSCORR":
                    ts_corrs = []
                    for t in range(fcst.time.size):
                        f_slice = fcst.isel(time=t)
                        o_slice = obs.isel(time=t)
                        corr = xs.pearson_r(o_slice, f_slice, dim=["lat", "lon"], skipna=True).values
                        ts_corrs.append(corr)
                    val = np.nanmean(ts_corrs)
                elif metric == "STCORR":
                    corr = xs.pearson_r(obs, fcst, dim="time", skipna=True)
                    weights = self.generate_coslat_weight(fcst)[0, :, :]
                    val = (corr * weights).mean(["lat", "lon"]).values
                else:
                    raise ValueError(f"Unsupported metric: {metric}")
                    
                results[season] = val

            ds_out = xr.Dataset({
                var: xr.DataArray(np.asarray([results[s] for s in season_list]),
                                  dims="season", coords={"season": season_list})
            })
            ds_out.to_netcdf(out_file)

    def _get_time_range_for_year(self, year):
        freq_map = {"3hourly": "3h", "6hourly": "6h", "monthly": "1MS"}
        return xr.cftime_range(f"{year}-01-01", f"{year}-12-31",
                               freq=freq_map[self.frequency], calendar="noleap")
        
    def _get_time_range_for_season(self, year, season):
        months, _ = self.seasons[season]
        freq = {"3hourly": "3h", "6hourly": "6h", "monthly": "1MS"}[self.frequency]

        dates = []
        for m in months:
            y = year
            # for DJF, December belongs to previous year
            if season == "DJF" and m == 12:
                y = year - 1
            dates.append(DatetimeNoLeap(y, m, 1))

        return xr.CFTimeIndex(dates)
        
    def _select_level(self, dr, target_plev):
        """
        Select vertical level based on varname suffix (e.g. U200 → 200 hPa).
        target_plev is expected in Pa, so it is converted to hPa here.
        Falls back to first level if no match.
        """
        # Convert target to hPa
        target_hpa = target_plev / 100.0
    
        for lev_dim in ["plev", "lev", "level"]:
            if lev_dim in dr.dims:
                lev = dr[lev_dim]
                lev_values = lev.values
    
                # Normalize lev_values to hPa
                units = getattr(lev, "units", "").lower()
                if "pa" in units and not "hpa" in units:
                    lev_values = lev_values / 100.0
                elif lev_values.max() > 2000:  # heuristic: probably Pa
                    lev_values = lev_values / 100.0
    
                if target_hpa is not None:
                    # Both in hPa now
                    idx = (abs(lev_values - target_hpa)).argmin()
                    dr = dr.isel({lev_dim: idx}, drop=True)
                    return dr
    
                # fallback: first level
                dr = dr.isel({lev_dim: 0}, drop=True)
                return dr
    
        # no level dimension found
        return dr
        
    def _parse_varname(self,varname):
        """
        Split varname into (base, level).
        Example: "U200" -> ("U", 20000) assuming Pa in dataset.
        Returns (base, None) if no level suffix found.
        """
        m = re.match(r"([A-Za-z]+)(\d+)$", varname)
        if m:
            base = m.group(1)
            level = int(m.group(2)) * 100  # assume hPa -> Pa
            return base, level
        return varname, None
        
    def _years_from_time_sub(self, time_sub, fallback_year=None):
        years = set()
        # Case 1: slice with .start/.stop datetimes
        if isinstance(time_sub, slice) and hasattr(time_sub, "start") and hasattr(time_sub, "stop"):
            for endpoint in (time_sub.start, time_sub.stop):
                if hasattr(endpoint, "year"):
                    years.add(int(endpoint.year))
        else:
            # Case 2: iterable of datetimes (e.g., CFTimeIndex) or single datetime
            try:
                for t in time_sub:
                    if hasattr(t, "year"):
                        years.add(int(t.year))
            except TypeError:
                # Not iterable → maybe a single datetime
                if hasattr(time_sub, "year"):
                    years.add(int(time_sub.year))
    
        if not years and fallback_year is not None:
            years = {int(fallback_year)}
        return sorted(years)
        
    def _set_time_to_midpoint(self,ds):
        if 'time_bnds' in ds.variables and 'time' in ds.coords:
            mid = ds['time_bnds'].mean(dim=ds['time_bnds'].dims[-1])  # average over nbnds
            ds = ds.assign_coords(time=mid)
        return ds
        
    def _normalize_time_to_month_start(self, ds):
        if "time" not in ds.coords:
            return ds
        new_times = []
        for t in ds["time"].values:
            # pandas/numpy datetime → use pandas
            if hasattr(t, "to_datetime64") or str(type(t)).endswith("Timestamp'>") or "datetime64" in str(type(t)):
                # convert to Month Start
                ts = xr.coding.cftimeindex.to_datetimeindex(xr.DataArray([t])).to_pandas()[0]
                new_times.append(ts.to_period('M').to_timestamp('MS'))
            else:
                # cftime (noleap, etc.) → reconstruct class with day=1
                cls = t.__class__
                new_times.append(cls(t.year, t.month, 1))
        ds = ds.assign_coords(time=("time", new_times))
        return ds
    
    def _read_model_data(self, exp, period, time_sub, regnam, var, vfac, data_dir, template, diag_print=False):
        # Determine which years to open (e.g., DJF spans year-1 and year)
        years_needed = self._years_from_time_sub(time_sub, fallback_year=period)
        paths = []
        for y in years_needed:
            pattern = os.path.join(data_dir, template.format(y))
            matches = sorted(glob.glob(pattern))
            for p in matches:
                if os.path.exists(p):
                    paths.append(p)
                else:
                    print(f"[WARN] Model file missing: {p}")
                
        if not paths:
            raise FileNotFoundError(f"No model files found for years={years_needed}")
            
        # ---- open & harmonize ----
        dm = xr.open_mfdataset(paths, combine="by_coords")
        
        region = self.define_region(regnam)
        lat_slice = slice(region[0][0], region[0][1])
        lon_slice = slice(region[1][0], region[1][1])
        base, target_plev = self._parse_varname(var)
        if base == "Z": base = 'Z3'

        for lev_dim in ["lev", "plev", "level"]:
            dm = self._select_level(dm, target_plev)
            
        if dm.lon.min() > -1.0:
            dm = dm.assign_coords(lon=((dm.lon + 180) % 360 - 180)).sortby('lon')
        dm = self._set_time_to_midpoint(dm)
        dm = self._normalize_time_to_month_start(dm)
        dm = dm.convert_calendar("noleap", use_cftime=True)
        ds = dm.sel(time=time_sub, method='nearest')
        ds = ds.sel(lat=lat_slice, lon=lon_slice)
        
        ds[var] = self.apply_unit_scaling_mod(var, ds[base], vfac)
        if diag_print :
            # --- Diagnostics: time coverage ---
            if "time" in ds.dims:
                print(time_sub)
                tmin = str(ds["time"].min().values)
                tmax = str(ds["time"].max().values)
                nt   = ds.sizes["time"]
                print(f"[DIAG] Model Time range = {tmin} → {tmax}  (n={nt} steps)")
            else:
                print("[DIAG] No Model time dimension found.")
                
        return ds

    def _read_reference_data(self, ref, period, time_sub, regnam, var, vfac, data_dir, template, diag_print=False):
        # ---- figure out which yearly files are needed ----
        years_needed = self._years_from_time_sub(time_sub,fallback_year=period)
        paths = []
        for y in years_needed:
            p = os.path.join(data_dir, template.format(y))
            if os.path.exists(p):
                paths.append(p)
            else:
                print(f"[WARN] Reference file missing: {p}")
    
        if not paths:
            raise FileNotFoundError(f"No reference files found for years={years_needed}")
    
        # ---- open & harmonize ----
        dr = xr.open_mfdataset(paths, combine="by_coords")
    
        # rename only if necessary (fixes: use dr.dims, not ds.dims)
        rename_dict = {}
        if "longitude" in dr.dims and "lon" not in dr.dims:
            rename_dict["longitude"] = "lon"
        if "latitude" in dr.dims and "lat" not in dr.dims:
            rename_dict["latitude"] = "lat"
        if rename_dict:
            dr = dr.rename(rename_dict)
    
        # parse varname and select pressure level
        base, target_plev = self._parse_varname(var)
        if base == "Z": base = 'Z3'
            
        for lev_dim in ["plev", "lev", "level"]:
            if lev_dim in dr.dims:
                dr = self._select_level(dr, target_plev)  # handles Pa→hPa internally
                break  # only once
                
        # normalize longitudes to [-180, 180]
        if "lon" in dr.coords and np.nanmin(dr["lon"]) >= 0:
            dr = dr.assign_coords(lon=((dr.lon + 180) % 360 - 180)).sortby("lon")
    
        # calendar handling then subsetting
        dr = dr.convert_calendar("noleap", use_cftime=True)
        dr = self._normalize_time_to_month_start(dr)

        # region/time subset (DJF will now work because both years are loaded)
        region = self.define_region(regnam)  # ((lat_min, lat_max), (lon_min, lon_max))
        if isinstance(time_sub, slice):
            dr = dr.sel(time=time_sub)  # exact range
        else:
            # if time_sub is (start, end) tuple or a single timestamp, handle gracefully
            dr = dr.sel(time=slice(*time_sub)) if isinstance(time_sub, (tuple, list)) else dr.sel(time=time_sub)
    
        dr = dr.sel(lat=slice(*region[0]), lon=slice(*region[1]))
    
        # apply unit scaling to the variable of interest
        dr[var] = self.apply_unit_scaling_obs(var, dr[base], vfac)

        if diag_print:
            # --- Diagnostics: time coverage ---
            if "time" in dr.dims:
                tmin = str(dr["time"].min().values)
                tmax = str(dr["time"].max().values)
                nt   = dr.sizes["time"]
                print(f"[DIAG] OBS Time range = {tmin} → {tmax}  (n={nt} steps)")
            else:
                print("[DIAG] No OBS time dimension found.")
            
        return dr

    @staticmethod
    def generate_coslat_weight(ds):
        weights = np.cos(np.deg2rad(ds.lat))
        _, weights = xr.broadcast(ds, weights)
        return weights

    @staticmethod
    def apply_unit_scaling_obs(var, da, vfac):
        if var in ['Z200', 'Z500', 'Z850']:
            return da * vfac / 9.80616
        elif var in ['SHFLX', 'TAUX', 'TAUY']:
            return da * vfac * -1.0
        elif var == 'LHFLX':
            return da * vfac * -2.501e6
        elif var in ['T200', 'T500', 'T850', 'TREFHT', 'TS']:
            return (da - 273.15) * vfac
        elif var == 'PRECT':
            return da * vfac / 3600.0 * 1000.0 * 86400.0
        else:
            return da * vfac

    @staticmethod
    def apply_unit_scaling_mod(var, da, vfac):
        if var in ['Z200', 'Z500', 'Z850']:
            return da * vfac / 9.80616
        elif var == 'PRECT':
            return da * vfac * 1000.0 * 86400.0
        elif var == 'LHFLX':
            return da * vfac * -2.501e6
        elif var in ['T200', 'T500', 'T850', 'TREFHT', 'TS']:
            return (da - 273.15) * vfac
        else:
            return da * vfac

    @staticmethod
    def define_region(regnam='global'):
        reg_dict = {
            'global': [(-90, 90), (-180, 180)],
            'Atlantic': [(5, 55), (-95, -40)],
            'CONUS': [(25, 50), (-125, -95)],
            'Antarctic': [(-90, -50), (-180, 180)],
            'PolarN': [(50, 90), (-180, 180)],
            'Greenland': [(60, 85), (-75, -10)]
        }
        return reg_dict[regnam]

    @staticmethod
    def extract_var_list():
        return {
            'U200':       {'alias': 'U200',     'unit': 'm s$^{-1}$',           'fscl': 1.0,    'min': -0.2, 'max': 0.2, 'nlev': 11},
            'U500':       {'alias': 'U500',     'unit': 'm s$^{-1}$',           'fscl': 1.0,    'min': -0.2, 'max': 0.2, 'nlev': 11},
            'U850':       {'alias': 'U850',     'unit': 'm s$^{-1}$',           'fscl': 1.0,    'min': -0.2, 'max': 0.2, 'nlev': 11},
            'V200':       {'alias': 'V200',     'unit': 'm s$^{-1}$',           'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'V500':       {'alias': 'V500',     'unit': 'm s$^{-1}$',           'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'V850':       {'alias': 'V850',     'unit': 'm s$^{-1}$',           'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'T850':       {'alias': 'T850',     'unit': '$^{o}$C',              'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'T200':       {'alias': 'T200',     'unit': '$^{o}$C',              'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'T500':       {'alias': 'T500',     'unit': '$^{o}$C',              'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'Q850':       {'alias': 'Q850',     'unit': 'g kg$^{-1}$',          'fscl': 1e3,    'min': -2,   'max': 2,   'nlev': 11},
            'Q200':       {'alias': 'Q200',     'unit': 'g kg$^{-1}$',          'fscl': 1e3,    'min': -2,   'max': 2,   'nlev': 11},
            'Q500':       {'alias': 'Q500',     'unit': 'g kg$^{-1}$',          'fscl': 1e3,    'min': -2,   'max': 2,   'nlev': 11},
            'OMEGA850':   {'alias': 'OMEGA850', 'unit': 'g kg$^{-1}$',          'fscl': 1e3,    'min': -2,   'max': 2,   'nlev': 11},
            'OMEGA200':   {'alias': 'OMEGA200', 'unit': 'g kg$^{-1}$',          'fscl': 1e3,    'min': -2,   'max': 2,   'nlev': 11},
            'OMEGA500':   {'alias': 'OMEGA500', 'unit': 'g kg$^{-1}$',          'fscl': 1e3,    'min': -2,   'max': 2,   'nlev': 11},            
            'Z850':       {'alias': 'Z850',     'unit': 'hectometer',           'fscl': 1e-2,   'min': -2,   'max': 2,   'nlev': 11},
            'Z200':       {'alias': 'Z200',     'unit': 'hectometer',           'fscl': 1e-2,   'min': -2,   'max': 2,   'nlev': 11},
            'Z500':       {'alias': 'Z500',     'unit': 'hectometer',           'fscl': 1e-2,   'min': -2,   'max': 2,   'nlev': 11},
        }

In [155]:
if __name__ == "__main__":
    # Set basic paths
    top_path  = "/pscratch/sd/z/zhan391/seacrogs_scratch"
    data_path = f"{top_path}/post_data"
    out_path  = "/pscratch/sd/z/zhan391/SEACROGS_project/paper_material/method_paper/fig_data/metric_data"
    os.makedirs(out_path, exist_ok=True)
    
    # Load model experiment metadata
    exp_json = f"/pscratch/sd/z/zhan391/seacrogs_scratch/post_data/scripts/ml_exp_info.json"
    with open(exp_json, "r") as f:
        exp_dict = json.load(f)
        
    # Time and frequency
    tstart = "2012-01"
    tend   = "2016-12"
    freq   = "monthly"
    regnam = "global"

    # Define reference (ERA5) metadata
    ref_dict = {
        "ERA5": {
            "run": "/pscratch/sd/z/zhan391/seacrogs_scratch/post_data/ERA5",
            "period": "200801_201712"
        }
    }
    
    # Template for locating model files
    path_template = f"{data_path}/%(CASENAME)/{freq}"

    # Metrics and variables to compute
    metrics = ["MEAN", "BIAS", "RMSE", "ACC", "TSCORR", "STCORR"]
    variables = None  # use all default variables from extract_var_list()

    # Initialize and run
    calculator = ErrorMetricCalculator(
        regnam=regnam,
        tstart=tstart,
        tend=tend,
        frequency=freq,
        model_list=list(exp_dict.keys()),
        ref_dict=ref_dict,
        exp_dict=exp_dict,
        path_in=path_template,
        out_path=out_path,
        var_list=variables,
        force=True
    )

    for metric in metrics:
        calculator.compute(metric)

Obs range: -16.133264541625977 to 84.33141326904297
Fcst range: -18.626991271972656 to 73.21190643310547
Obs range: -15.872729301452637 to 71.07432556152344
Fcst range: -15.181425094604492 to 59.3435173034668
Obs range: -26.388538360595703 to 61.55156707763672
Fcst range: -28.120893478393555 to 63.29940414428711
Obs range: -20.89841651916504 to 64.02476501464844
Fcst range: -19.400861740112305 to 68.6537857055664
Obs range: -26.388538360595703 to 84.33141326904297
Fcst range: -28.120893478393555 to 76.12449645996094
Obs range: -16.133264541625977 to 84.33141326904297
Fcst range: -22.00020408630371 to 78.05396270751953
Obs range: -15.872729301452637 to 71.07432556152344
Fcst range: -13.184670448303223 to 66.61177062988281
Obs range: -26.388538360595703 to 61.55156707763672
Fcst range: -25.31246566772461 to 60.4326171875
Obs range: -20.89841651916504 to 64.02476501464844
Fcst range: -17.936479568481445 to 59.72868347167969
Obs range: -26.388538360595703 to 84.33141326904297
Fcst range: -


KeyboardInterrupt

